In [2]:
import time
import random
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
from IPython.display import display

# ============================================================
# PART 1 — REDDIT SCRAPER FOR CLAUDE DISCOURSE
# Public data only
# Jupyter-friendly
# Includes posts, comments, engagement, and audience proxies
# ============================================================

# ----------------------------
# CONFIG
# ----------------------------
KEYWORDS = [
    "claude",
    "anthropic",
    "claude ai",
    "claude sonnet",
    "claude opus",
    "claude 3",
    "claude 3.5",
    "claude 3.7"
]

SUBREDDITS = [
    "Anthropic",
    "ClaudeAI",
    "singularity",
    "MachineLearning",
    "artificial",
    "ChatGPT",
    "LocalLLaMA",
    "OpenAI"
]

LOOKBACK_DAYS = 180
RECENT_LIMIT_PER_SUB = 150
TOP_LIMIT_PER_SUB = 150
MAX_COMMENTS_PER_POST = 100

POSTS_OUTPUT = "reddit_posts_rich_metrics.csv"
COMMENTS_OUTPUT = "reddit_comments_rich_metrics.csv"

# ----------------------------
# SCRAPER
# ----------------------------
class RedditPublicScraper:
    BASE_URL = "https://www.reddit.com"

    def __init__(self, sleep_seconds=2.0, timeout=30, max_retries=5):
        self.sleep_seconds = sleep_seconds
        self.timeout = timeout
        self.max_retries = max_retries

        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (compatible; HackNURedditResearch/1.0)"
        })

    def _request_json(self, url, params=None):
        params = params or {}
        params["raw_json"] = 1

        last_error = None

        for attempt in range(1, self.max_retries + 1):
            try:
                response = self.session.get(url, params=params, timeout=self.timeout)

                if response.status_code == 200:
                    time.sleep(self.sleep_seconds + random.uniform(0.2, 0.5))
                    return response.json()

                if response.status_code in (429, 500, 502, 503, 504):
                    wait = min(2 ** attempt, 30)
                    print(f"Retryable HTTP {response.status_code}. Sleeping {wait}s...")
                    time.sleep(wait)
                    continue

                response.raise_for_status()

            except requests.RequestException as e:
                last_error = e
                wait = min(2 ** attempt, 30)
                print(f"Attempt {attempt} failed: {e}. Sleeping {wait}s...")
                time.sleep(wait)

        raise RuntimeError(f"Request failed after retries. Last error: {last_error}")

    def fetch_subreddit_about(self, subreddit):
        url = f"{self.BASE_URL}/r/{subreddit}/about.json"
        payload = self._request_json(url)
        data = payload.get("data", {})
        return {
            "subreddit": data.get("display_name"),
            "subreddit_title": data.get("title"),
            "subreddit_description": data.get("public_description"),
            "subreddit_subscribers": data.get("subscribers"),
            "subreddit_active_accounts": data.get("active_user_count"),
            "subreddit_created_utc": data.get("created_utc"),
            "subreddit_over18": data.get("over18"),
        }

    def fetch_subreddit_posts(self, subreddit, sort="new", total_limit=100, time_filter="all"):
        all_posts = []
        after = None
        fetched = 0

        while fetched < total_limit:
            batch_limit = min(100, total_limit - fetched)
            url = f"{self.BASE_URL}/r/{subreddit}/{sort}.json"

            params = {
                "limit": batch_limit,
                "after": after,
                "count": fetched
            }

            if sort in {"top", "controversial"}:
                params["t"] = time_filter

            payload = self._request_json(url, params=params)
            listing = payload.get("data", {})
            children = listing.get("children", [])

            if not children:
                break

            for child in children:
                d = child.get("data", {})

                all_posts.append({
                    "post_id": d.get("id"),
                    "fullname": d.get("name"),
                    "subreddit": d.get("subreddit"),
                    "title": d.get("title"),
                    "selftext": d.get("selftext"),
                    "author": d.get("author"),
                    "created_utc": d.get("created_utc"),
                    "score": d.get("score"),  # best public proxy for likes/upvotes
                    "upvote_ratio": d.get("upvote_ratio"),
                    "num_comments": d.get("num_comments"),
                    "total_awards_received": d.get("total_awards_received"),
                    "num_crossposts": d.get("num_crossposts"),
                    "view_count": d.get("view_count"),  # often missing/null
                    "is_video": d.get("is_video"),
                    "post_hint": d.get("post_hint"),
                    "domain": d.get("domain"),
                    "url": d.get("url"),
                    "permalink": f"{self.BASE_URL}{d.get('permalink', '')}",
                    "thumbnail": d.get("thumbnail"),
                    "link_flair_text": d.get("link_flair_text"),
                    "is_self": d.get("is_self"),
                    "over_18": d.get("over_18"),
                    "spoiler": d.get("spoiler"),
                    "stickied": d.get("stickied"),
                    "locked": d.get("locked"),
                    "distinguished": d.get("distinguished"),
                    "sort_mode": sort,
                    "time_filter": time_filter
                })

            fetched += len(children)
            after = listing.get("after")

            if not after:
                break

        return all_posts

    def fetch_post_comments(self, permalink, max_comments=100, sort="top"):
        if permalink.startswith("http"):
            url = permalink.rstrip("/") + ".json"
        else:
            url = f"{self.BASE_URL}{permalink.rstrip('/')}.json"

        params = {
            "limit": 500,
            "sort": sort,
            "depth": 10
        }

        payload = self._request_json(url, params=params)

        if not isinstance(payload, list) or len(payload) < 2:
            return []

        comments_listing = payload[1]
        results = []

        def walk(nodes):
            nonlocal results

            for node in nodes:
                kind = node.get("kind")
                data = node.get("data", {})

                if kind == "t1":
                    results.append({
                        "comment_id": data.get("id"),
                        "fullname": data.get("name"),
                        "parent_id": data.get("parent_id"),
                        "link_id": data.get("link_id"),
                        "author": data.get("author"),
                        "body": data.get("body"),
                        "score": data.get("score"),
                        "total_awards_received": data.get("total_awards_received"),
                        "controversiality": data.get("controversiality"),
                        "created_utc": data.get("created_utc"),
                        "is_submitter": data.get("is_submitter"),
                        "distinguished": data.get("distinguished"),
                        "stickied": data.get("stickied"),
                        "permalink": f"{self.BASE_URL}{data.get('permalink', '')}",
                    })

                    if len(results) >= max_comments:
                        return

                    replies = data.get("replies")
                    if isinstance(replies, dict):
                        children = replies.get("data", {}).get("children", [])
                        walk(children)

                        if len(results) >= max_comments:
                            return

                elif kind == "more":
                    continue

        root_nodes = comments_listing.get("data", {}).get("children", [])
        walk(root_nodes)

        return results[:max_comments]


# ----------------------------
# HELPERS
# ----------------------------
def normalize_text(x):
    if pd.isna(x) or x is None:
        return ""
    return str(x).strip().lower()

def contains_keywords(title, selftext, keywords):
    text = f"{normalize_text(title)} {normalize_text(selftext)}"
    return any(k.lower() in text for k in keywords)

def now_utc():
    return datetime.now(timezone.utc)

def cutoff_date(days):
    return now_utc() - timedelta(days=days)


# ----------------------------
# STEP 1: FETCH SUBREDDIT META
# ----------------------------
scraper = RedditPublicScraper(sleep_seconds=2.0, timeout=30, max_retries=5)

subreddit_meta = []

for sub in SUBREDDITS:
    try:
        meta = scraper.fetch_subreddit_about(sub)
        subreddit_meta.append(meta)
        print(f"Loaded subreddit meta for r/{sub}")
    except Exception as e:
        print(f"Failed subreddit meta for r/{sub}: {e}")

subreddit_meta_df = pd.DataFrame(subreddit_meta)


# ----------------------------
# STEP 2: FETCH POSTS
# Strategy:
# - recent posts via "new"
# - high-engagement posts via "top" over year
# - final dataset filtered to last 6 months
# ----------------------------
all_posts = []

for sub in SUBREDDITS:
    try:
        print(f"\nFetching recent posts from r/{sub} ...")
        recent_posts = scraper.fetch_subreddit_posts(
            subreddit=sub,
            sort="new",
            total_limit=RECENT_LIMIT_PER_SUB,
            time_filter="all"
        )
        all_posts.extend(recent_posts)
        print(f"Collected {len(recent_posts)} recent posts")
    except Exception as e:
        print(f"Failed recent fetch for r/{sub}: {e}")

    try:
        print(f"Fetching top posts from r/{sub} over year ...")
        top_posts = scraper.fetch_subreddit_posts(
            subreddit=sub,
            sort="top",
            total_limit=TOP_LIMIT_PER_SUB,
            time_filter="year"
        )
        all_posts.extend(top_posts)
        print(f"Collected {len(top_posts)} top/year posts")
    except Exception as e:
        print(f"Failed top/year fetch for r/{sub}: {e}")

posts_df = pd.DataFrame(all_posts)

if posts_df.empty:
    raise ValueError("No posts collected.")

posts_df = posts_df.drop_duplicates(subset=["post_id"]).reset_index(drop=True)

posts_df["created_datetime"] = pd.to_datetime(
    posts_df["created_utc"], unit="s", utc=True, errors="coerce"
)

posts_df["keyword_match"] = posts_df.apply(
    lambda row: contains_keywords(row.get("title"), row.get("selftext"), KEYWORDS),
    axis=1
)

posts_df = posts_df[posts_df["keyword_match"]].copy()

cutoff = cutoff_date(LOOKBACK_DAYS)
posts_df = posts_df[posts_df["created_datetime"] >= cutoff].copy()

# Join subreddit-level audience proxies
if not subreddit_meta_df.empty:
    posts_df = posts_df.merge(
        subreddit_meta_df[[
            "subreddit",
            "subreddit_subscribers",
            "subreddit_active_accounts",
            "subreddit_title",
            "subreddit_description"
        ]],
        on="subreddit",
        how="left"
    )

# Derived metrics
posts_df["engagement_score_simple"] = (
    posts_df["score"].fillna(0)
    + posts_df["num_comments"].fillna(0)
    + (posts_df["total_awards_received"].fillna(0) * 3)
    + (posts_df["num_crossposts"].fillna(0) * 2)
)

posts_df["comments_per_score"] = posts_df["num_comments"] / posts_df["score"].replace(0, pd.NA)
posts_df["score_per_subscriber"] = posts_df["score"] / posts_df["subreddit_subscribers"].replace(0, pd.NA)

posts_df["scraped_at_utc"] = now_utc().isoformat()
posts_df["lookback_days"] = LOOKBACK_DAYS

print("\nPOSTS READY")
print("Posts in final 6-month dataset:", len(posts_df))


# ----------------------------
# STEP 3: FETCH COMMENTS
# ----------------------------
all_comments = []

for idx, row in posts_df.iterrows():
    try:
        comments = scraper.fetch_post_comments(
            permalink=row["permalink"],
            max_comments=MAX_COMMENTS_PER_POST,
            sort="top"
        )

        for c in comments:
            c["post_id"] = row["post_id"]
            c["post_title"] = row["title"]
            c["post_subreddit"] = row["subreddit"]
            c["post_score"] = row["score"]
            c["post_num_comments"] = row["num_comments"]
            c["post_subreddit_subscribers"] = row.get("subreddit_subscribers")

        all_comments.extend(comments)

        if (idx + 1) % 10 == 0:
            print(f"Processed comments for {idx + 1} posts")
    except Exception as e:
        print(f"Failed comments for post {row['post_id']}: {e}")

comments_df = pd.DataFrame(all_comments)

if not comments_df.empty:
    comments_df = comments_df.drop_duplicates(subset=["comment_id"]).reset_index(drop=True)
    comments_df["created_datetime"] = pd.to_datetime(
        comments_df["created_utc"], unit="s", utc=True, errors="coerce"
    )
    comments_df["scraped_at_utc"] = now_utc().isoformat()
    comments_df["lookback_days"] = LOOKBACK_DAYS

print("\nCOMMENTS READY")
print("Comments collected:", len(comments_df))


# ----------------------------
# STEP 4: SAVE
# ----------------------------
posts_df.to_csv(POSTS_OUTPUT, index=False)
comments_df.to_csv(COMMENTS_OUTPUT, index=False)

print("\nFILES SAVED")
print("-", POSTS_OUTPUT)
print("-", COMMENTS_OUTPUT)


# ----------------------------
# STEP 5: QUICK REVIEW
# ----------------------------
print("\nPOSTS SHAPE:", posts_df.shape)
print("COMMENTS SHAPE:", comments_df.shape)

if not posts_df.empty:
    print("\nTimeline range:")
    print("Earliest post:", posts_df["created_datetime"].min())
    print("Latest post:  ", posts_df["created_datetime"].max())

    print("\nTop 10 posts by engagement score:")
    display(
        posts_df.sort_values("engagement_score_simple", ascending=False)[[
            "created_datetime",
            "subreddit",
            "title",
            "score",
            "num_comments",
            "total_awards_received",
            "num_crossposts",
            "subreddit_subscribers",
            "view_count",
            "engagement_score_simple",
            "permalink"
        ]].head(10)
    )

print("\nPosts preview:")
display(posts_df.head(10))

print("\nComments preview:")
display(comments_df.head(10))

Loaded subreddit meta for r/Anthropic
Loaded subreddit meta for r/ClaudeAI
Loaded subreddit meta for r/singularity
Loaded subreddit meta for r/MachineLearning
Loaded subreddit meta for r/artificial
Loaded subreddit meta for r/ChatGPT
Loaded subreddit meta for r/LocalLLaMA
Loaded subreddit meta for r/OpenAI

Fetching recent posts from r/Anthropic ...
Collected 150 recent posts
Fetching top posts from r/Anthropic over year ...
Collected 150 top/year posts

Fetching recent posts from r/ClaudeAI ...
Collected 150 recent posts
Fetching top posts from r/ClaudeAI over year ...
Collected 150 top/year posts

Fetching recent posts from r/singularity ...
Collected 150 recent posts
Fetching top posts from r/singularity over year ...
Collected 150 top/year posts

Fetching recent posts from r/MachineLearning ...
Collected 150 recent posts
Fetching top posts from r/MachineLearning over year ...
Collected 150 top/year posts

Fetching recent posts from r/artificial ...
Collected 150 recent posts
Fetchi

,created_datetime,subreddit,title,score,num_comments,total_awards_received,num_crossposts,subreddit_subscribers,view_count,engagement_score_simple,permalink
543,2026-02-28 15:51:51+00:00,ChatGPT,"Cancel your ChatGPT Plus, burn their compute o...",29929,1996,0,4,11428303,None,31933,https://www.reddit.com/r/ChatGPT/comments/1rh6...
544,2026-02-27 21:37:41+00:00,ChatGPT,"Hey, OpenAI: Watch and f****** learn. This is ...",18132,1442,0,4,11428303,None,19582,https://www.reddit.com/r/ChatGPT/comments/1rgj...
459,2026-02-28 18:13:22+00:00,singularity,Cancel your Chatgpt subscriptions and pick up ...,8512,826,0,2,3869498,None,9342,https://www.reddit.com/r/singularity/comments/...
334,2026-04-03 18:19:27+00:00,ClaudeAI,Taught Claude to talk like a caveman to use 75...,8208,376,0,7,701292,None,8598,https://www.reddit.com/r/ClaudeAI/comments/1sb...
460,2026-02-12 22:43:23+00:00,singularity,"Anthropic raises $30B, Elon crashes out",6993,946,0,3,3869498,None,7945,https://www.reddit.com/r/singularity/comments/...
335,2026-02-17 07:28:40+00:00,ClaudeAI,Good job Anthropic 👏🏻 you just became the top ...,7362,429,0,3,701292,None,7797,https://www.reddit.com/r/ClaudeAI/comments/1r6...
337,2026-02-27 20:40:27+00:00,ClaudeAI,"Outside Anthropic Office in SF ""Thank You""",7063,204,0,3,701292,None,7273,https://www.reddit.com/r/ClaudeAI/comments/1rg...
336,2026-03-01 22:46:43+00:00,ClaudeAI,Claude’s extended thinking found out about Ira...,7080,178,0,2,701292,None,7262,https://www.reddit.com/r/ClaudeAI/comments/1ri...
339,2026-03-26 08:08:06+00:00,ClaudeAI,25 years. Multiple specialists. Zero answers. ...,5588,1087,0,11,701292,None,6697,https://www.reddit.com/r/ClaudeAI/comments/1s4...
338,2026-02-28 23:01:47+00:00,ClaudeAI,Claude has overtaken ChatGPT in the Apple App ...,6427,219,0,2,701292,None,6650,https://www.reddit.com/r/ClaudeAI/comments/1rh...



Posts preview:


,post_id,fullname,subreddit,title,selftext,author,created_utc,score,upvote_ratio,num_comments,...,keyword_match,subreddit_subscribers,subreddit_active_accounts,subreddit_title,subreddit_description,engagement_score_simple,comments_per_score,score_per_subscriber,scraped_at_utc,lookback_days
0,1scfapf,t3_1scfapf,Anthropic,Usage Limits Are Even Getting Predtictive,Pro user. I ran a claude research request this...,modbroccoli,1.775324e+09,8,0.75,2,...,True,118311,None,Anthropic,Anthropic (and Claude) Community,10,0.25,0.000068,2026-04-04T18:22:22.664839+00:00,180
1,1scf5mb,t3_1scf5mb,Anthropic,Question on Privacy and Claude,"So I am technically savvy, and have watched th...",laternerdz,1.775324e+09,0,0.50,1,...,True,118311,None,Anthropic,Anthropic (and Claude) Community,1,<NA>,0.000000,2026-04-04T18:22:22.664839+00:00,180
2,1scc7ed,t3_1scc7ed,Anthropic,"​""Co​-​Authored​-​By: ​Claude ​Opus 4​.6""",I'm on Xcode and I asked Claude Code to commit...,Delicious_Volume3306,1.775317e+09,0,0.50,6,...,True,118311,None,Anthropic,Anthropic (and Claude) Community,6,<NA>,0.000000,2026-04-04T18:22:22.664839+00:00,180
3,1scbzuf,t3_1scbzuf,Anthropic,What do you do when you get 500 errors from an...,"I'm a $200 max pro member, I love their [https...",EggplantFunTime,1.775316e+09,4,0.83,4,...,True,118311,None,Anthropic,Anthropic (and Claude) Community,8,1.0,0.000034,2026-04-04T18:22:22.664839+00:00,180
4,1sca4i0,t3_1sca4i0,Anthropic,Thanks Anthropic! Claude #1.,,RobinInPH,1.775312e+09,0,0.30,5,...,True,118311,None,Anthropic,Anthropic (and Claude) Community,5,<NA>,0.000000,2026-04-04T18:22:22.664839+00:00,180
5,1sc7l9z,t3_1sc7l9z,Anthropic,Cannot buy credits for API usage.,"Hey,\n\nI hit the daily limit of Claude Code a...",Tex-Twil,1.775305e+09,1,1.00,0,...,True,118311,None,Anthropic,Anthropic (and Claude) Community,1,0.0,0.000008,2026-04-04T18:22:22.664839+00:00,180
6,1sc78t5,t3_1sc78t5,Anthropic,Hands-free programming with Claude Code: what’...,,alvarolb84,1.775303e+09,1,0.67,0,...,True,118311,None,Anthropic,Anthropic (and Claude) Community,1,0.0,0.000008,2026-04-04T18:22:22.664839+00:00,180
7,1sc6xve,t3_1sc6xve,Anthropic,Can't access Fin the AI support agent!!!,So here's the timeline of my story:\n\n28/3 - ...,Litvinsev,1.775302e+09,0,0.33,2,...,True,118311,None,Anthropic,Anthropic (and Claude) Community,2,<NA>,0.000000,2026-04-04T18:22:22.664839+00:00,180
8,1sc6we9,t3_1sc6we9,Anthropic,Could someone explain the usage limits between...,I went to a friend's house today to work on so...,SIREN-25,1.775302e+09,1,0.67,12,...,True,118311,None,Anthropic,Anthropic (and Claude) Community,13,12.0,0.000008,2026-04-04T18:22:22.664839+00:00,180
9,1sc64m4,t3_1sc64m4,Anthropic,Are we about to lose the claude-cli -p behavio...,"I don't use openclaw, I don't use opencode, bu...",constarx,1.775300e+09,11,0.83,14,...,True,118311,None,Anthropic,Anthropic (and Claude) Community,25,1.272727,0.000093,2026-04-04T18:22:22.664839+00:00,180



Comments preview:


,comment_id,fullname,parent_id,link_id,author,body,score,total_awards_received,controversiality,created_utc,...,permalink,post_id,post_title,post_subreddit,post_score,post_num_comments,post_subreddit_subscribers,created_datetime,scraped_at_utc,lookback_days
0,oeaobyi,t1_oeaobyi,t3_1scfapf,t3_1scfapf,Impressive_Care_7558,They were just going to do this later when you...,1,0,0,1.775326e+09,...,https://www.reddit.com/r/Anthropic/comments/1s...,1scfapf,Usage Limits Are Even Getting Predtictive,Anthropic,8,2,118311,2026-04-04 18:14:36+00:00,2026-04-04T19:21:03.307544+00:00,180
1,oe9rv3l,t1_oe9rv3l,t3_1scc7ed,t3_1scc7ed,Southern_Reference23,It's been there since the first day,7,0,0,1.775317e+09,...,https://www.reddit.com/r/Anthropic/comments/1s...,1scc7ed,"​""Co​-​Authored​-​By: ​Claude ​Opus 4​.6""",Anthropic,0,6,118311,2026-04-04 15:36:46+00:00,2026-04-04T19:21:03.307544+00:00,180
2,oea7hch,t1_oea7hch,t3_1scc7ed,t3_1scc7ed,PoweredByte,"""attribution"": {\n\n""commit"": """",\n\n""pr"": ""...",4,0,0,1.775322e+09,...,https://www.reddit.com/r/Anthropic/comments/1s...,1scc7ed,"​""Co​-​Authored​-​By: ​Claude ​Opus 4​.6""",Anthropic,0,6,118311,2026-04-04 16:53:26+00:00,2026-04-04T19:21:03.307544+00:00,180
3,oe9zhyr,t1_oe9zhyr,t3_1scc7ed,t3_1scc7ed,profpendog,"Well, the commit message is part of the commit...",2,0,0,1.775319e+09,...,https://www.reddit.com/r/Anthropic/comments/1s...,1scc7ed,"​""Co​-​Authored​-​By: ​Claude ​Opus 4​.6""",Anthropic,0,6,118311,2026-04-04 16:14:08+00:00,2026-04-04T19:21:03.307544+00:00,180
4,oe9zuar,t1_oe9zuar,t3_1scc7ed,t3_1scc7ed,larowin,"Please, everyone, at least scan the documentat...",1,0,0,1.775319e+09,...,https://www.reddit.com/r/Anthropic/comments/1s...,1scc7ed,"​""Co​-​Authored​-​By: ​Claude ​Opus 4​.6""",Anthropic,0,6,118311,2026-04-04 16:15:48+00:00,2026-04-04T19:21:03.307544+00:00,180
5,oe9xkmr,t1_oe9xkmr,t3_1scc7ed,t3_1scc7ed,Hungry_Age5375,"Yep, new behavior. AI tools claiming credit th...",-1,0,0,1.775319e+09,...,https://www.reddit.com/r/Anthropic/comments/1s...,1scc7ed,"​""Co​-​Authored​-​By: ​Claude ​Opus 4​.6""",Anthropic,0,6,118311,2026-04-04 16:04:38+00:00,2026-04-04T19:21:03.307544+00:00,180
6,oe9ygxj,t1_oe9ygxj,t1_oe9xkmr,t3_1scc7ed,fynn34,"This is not new behavior, it’s been baked into...",2,0,0,1.775319e+09,...,https://www.reddit.com/r/Anthropic/comments/1s...,1scc7ed,"​""Co​-​Authored​-​By: ​Claude ​Opus 4​.6""",Anthropic,0,6,118311,2026-04-04 16:09:03+00:00,2026-04-04T19:21:03.307544+00:00,180
7,oea1kxd,t1_oea1kxd,t3_1scbzuf,t3_1scbzuf,Hungry_Age5375,Been there. This is why I configure backup pro...,2,0,0,1.775320e+09,...,https://www.reddit.com/r/Anthropic/comments/1s...,1scbzuf,What do you do when you get 500 errors from an...,Anthropic,4,4,118311,2026-04-04 16:24:23+00:00,2026-04-04T19:21:03.307544+00:00,180
8,oea7s3c,t1_oea7s3c,t3_1scbzuf,t3_1scbzuf,bishopLucas,Stand up and take a walk.,1,0,0,1.775322e+09,...,https://www.reddit.com/r/Anthropic/comments/1s...,1scbzuf,What do you do when you get 500 errors from an...,Anthropic,4,4,118311,2026-04-04 16:54:52+00:00,2026-04-04T19:21:03.307544+00:00,180
9,oe9pupz,t1_oe9pupz,t3_1scbzuf,t3_1scbzuf,ContestStreet,Just report and move on. Their entire customer...,0,0,0,1.775316e+09,...,https://www.reddit.com/r/Anthropic/comments/1s...,1scbzuf,What do you do when you get 500 errors from an...,Anthropic,4,4,118311,2026-04-04 15:26:54+00:00,2026-04-04T19:21:03.307544+00:00,180
